# Day 3 - Student Performance Prediction

Predicting a student's final grade (A/B/C/F) from study habits, attendance, previous scores, and other factors.
This is a multiclass classification problem.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

%matplotlib inline

## 1. Load Data

In [ ]:
df = pd.read_csv('dataset/students.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('Grade distribution:')
print(df['grade'].value_counts().sort_index())

## 2. EDA

In [ ]:
colors = {'A': '#4CAF50', 'B': '#2196F3', 'C': '#FF9800', 'F': '#F44336'}

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, feature in zip(axes.flatten(),
                        ['study_hours_per_day', 'attendance_percent',
                         'previous_score', 'sleep_hours']):
    for grade, color in colors.items():
        subset = df[df['grade'] == grade]
        ax.hist(subset[feature], alpha=0.6, label=grade, color=color, bins=20)
    ax.set_title(feature)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Grade')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
numeric_cols = df.select_dtypes(include=np.number).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

## 3. Train / Test Split

In [ ]:
feature_cols = [
    'study_hours_per_day', 'attendance_percent', 'previous_score',
    'sleep_hours', 'assignments_completed', 'parental_education_level',
    'internet_access', 'tutoring'
]

grade_map = {'A': 3, 'B': 2, 'C': 1, 'F': 0}
X = df[feature_cols]
y = df['grade'].map(grade_map)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train:', X_train.shape)
print('Test: ', X_test.shape)

## 4. Model Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    print(f'{name}  Accuracy: {acc:.4f}\n')
    print(classification_report(y_test, preds, target_names=['F','C','B','A'], zero_division=0))

## 5. Best Model - Confusion Matrix

In [ ]:
best = models['Logistic Regression']
preds = best.predict(X_test_scaled)

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['F','C','B','A'],
            yticklabels=['F','C','B','A'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

In [ ]:
rf = models['Random Forest']
importances = rf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(9, 5))
plt.bar([feature_cols[i] for i in sorted_idx], importances[sorted_idx], color='#2196F3')
plt.title('Feature Importance - Random Forest')
plt.ylabel('Importance')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

# study_hours and previous_score dominate

## Wrap-up

Logistic Regression outperformed both Random Forest and Gradient Boosting on this dataset with 70.5% accuracy. This happens when the decision boundaries between classes are roughly linear.

The hardest predictions are B vs C — students in the middle of the distribution are genuinely hard to separate. F and A students are predicted more reliably.

Most important features: study_hours_per_day and previous_score.